In [1]:
import pandas as pd
import numpy as np

In [2]:
rides_df = pd.read_excel("../data/PMPML_LIR_ML_Dataset.xlsx")
schema_df = pd.read_excel("../data/LIR_DB_Schema.xlsx")

print("Rides:", rides_df.shape)
print("Schema:", schema_df.shape)

Rides: (14, 7)
Schema: (100000, 14)


In [3]:
rides_df = pd.read_excel("../data/PMPML_LIR_ML_Dataset.xlsx", sheet_name="rides")
schema_df = pd.read_excel("../data/LIR_DB_Schema.xlsx", sheet_name="Shifts")

In [4]:
rides_df.rename(columns={
    "driverId": "driver_id",
    "rideKm": "ride_km",
    "durationMin": "duration_min",
    "requestedAt": "requested_at",
    "pickupLat": "pickup_lat",
    "pickupLng": "pickup_lng",
    "dropLat": "drop_lat",
    "dropLng": "drop_lng"
}, inplace=True)

schema_df.rename(columns={
    "driverId": "driver_id",
    "clockInTime": "shift_start",
    "clockOutTime": "shift_end"
}, inplace=True)

In [11]:
rides_df["requested_at"] = pd.to_datetime(rides_df["requested_at"])
schema_df["shift_start"] = pd.to_datetime(schema_df["shift_start"])
schema_df["shift_end"] = pd.to_datetime(schema_df["shift_end"])

In [12]:
# Merge
df = rides_df.merge(schema_df, on="driver_id", how="left")

print("Before cleaning:", df.shape)

# ⚠️ Only drop rows where IMPORTANT columns missing
df = df.dropna(subset=[
    "ride_km",
    "duration_min",
    "fare",
    "requested_at"
])

# If shift_end missing → fill dummy (VERY IMPORTANT)
df["shift_end"] = df["shift_end"].fillna(
    df["requested_at"] + pd.Timedelta(hours=8)
)

print("After cleaning:", df.shape)

Before cleaning: (2000, 51)
After cleaning: (2000, 51)


In [13]:
df["hour"] = df["requested_at"].dt.hour

df["remaining_time"] = (
    (df["shift_end"] - df["requested_at"]).dt.total_seconds() / 60
)

df["distance_proxy"] = df["ride_km"]

df["speed"] = df["ride_km"] / (df["duration_min"] + 1)
df["fare_per_km"] = df["fare"] / (df["ride_km"] + 1)

df["efficiency"] = df["fare"] / (df["duration_min"] + 1)

In [14]:
features = [
    "ride_km",
    "duration_min",
    "fare",
    "hour",
    "remaining_time",
    "distance_proxy",
    "speed",
    "fare_per_km"
]

X = df[features]
y = df["efficiency"]

print("Final dataset:", X.shape)

Final dataset: (2000, 8)


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

print("🔥 Model trained")

🔥 Model trained


In [17]:
from sklearn.metrics import mean_squared_error

y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("🔥 FINAL RMSE:", rmse)

🔥 FINAL RMSE: 0.16209276606063736


In [18]:
import joblib

joblib.dump(model, "../models/xgb_model.pkl")

print("🔥 FINAL MODEL SAVED SUCCESSFULLY")

🔥 FINAL MODEL SAVED SUCCESSFULLY
